# YOLOv8 Experiments: GOST Stamp Detection

**Цель:** Обучить YOLOv8 для детекции штампов на строительных чертежах.

**Данные:** 500 synthetic (train) + 49 real (val)

**Метрики:** IoU, Precision, Recall, F1 на 49 реальных изображениях

**Подход:** Single training run, yolov8n, 50 epochs, CPU (Colab)


In [ ]:
import sys
sys.path.insert(0, "../src")

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

from evaluation.metrics import DetectionResult, bbox_iou, yolo_to_pixel, compute_metrics, print_metrics
from data.loader import load_image_and_labels

PROJECT_DIR = Path("../")
DATA_DIR = PROJECT_DIR / "data"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
ARTIFACTS_DIR.mkdir(exist_ok=True)
(ARTIFACTS_DIR / "models").mkdir(exist_ok=True)
(ARTIFACTS_DIR / "metrics").mkdir(exist_ok=True)
(ARTIFACTS_DIR / "figures").mkdir(exist_ok=True)

IMAGE_TEST_DIR = DATA_DIR / "images" / "test"
LABEL_TEST_DIR = DATA_DIR / "labels" / "test"

print(f"Test images: {len(list(IMAGE_TEST_DIR.glob('*.png'))) + len(list(IMAGE_TEST_DIR.glob('*.jpg')))}")
print(f"Test labels: {len(list(LABEL_TEST_DIR.glob('*.txt')))}")

## Training

Запускаем YOLOv8n на данных из `gost_stamp.yaml`.

In [ ]:
import time
start = time.time()

model = YOLO("yolov8n.pt")

results = model.train(
    data=str(DATA_DIR / "gost_stamp.yaml"),
    epochs=50,
    imgsz=640,
    batch=16,
    device="cpu",
    project=str(ARTIFACTS_DIR / "yolo"),
    name="exp01",
    verbose=True,
    save=True,
    plots=True,
)

elapsed = time.time() - start
print(f"\nTraining time: {elapsed/60:.1f} minutes")

## Evaluation on 49 Real Images

Загружаем лучшие веса и оцениваем на тестовой выборке.

In [ ]:
best_weights = ARTIFACTS_DIR / "yolo" / "exp01" / "weights" / "best.pt"

if best_weights.exists():
    model = YOLO(str(best_weights))
    print(f"Loaded weights from {best_weights}")
else:
    print(f"Weights not found at {best_weights}, using last.pt")
    last_weights = ARTIFACTS_DIR / "yolo" / "exp01" / "weights" / "last.pt"
    model = YOLO(str(last_weights))

all_images = sorted(IMAGE_TEST_DIR.glob("*.png")) + sorted(IMAGE_TEST_DIR.glob("*.jpg"))
print(f"Evaluating on {len(all_images)} images")

results_list = []
for img_path in all_images:
    img, labels = load_image_and_labels(img_path, LABEL_TEST_DIR)
    h, w = img.shape[:2]
    
    gt_bbox = yolo_to_pixel(tuple(labels[0]), w, h) if len(labels) > 0 else None
    
    preds = model(img, conf=0.1, verbose=False)
    if preds[0].boxes and len(preds[0].boxes) > 0:
        box = preds[0].boxes[0]
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        pred_bbox = (x1, y1, x2 - x1, y2 - y1)
    else:
        pred_bbox = None
    
    iou = bbox_iou(pred_bbox, gt_bbox) if pred_bbox and gt_bbox else 0.0
    
    results_list.append(DetectionResult(
        image_name=img_path.name,
        gt_bbox=gt_bbox,
        pred_bbox=pred_bbox,
        iou=iou,
        found=pred_bbox is not None
    ))

metrics = compute_metrics(results_list, iou_threshold=0.5)
print_metrics(metrics, prefix="YOLO ")

## IoU Threshold Analysis

In [ ]:
print("IoU @ different thresholds:")
for thresh in [0.3, 0.5, 0.75]:
    m = compute_metrics(results_list, iou_threshold=thresh)
    print(f"  IoU >= {thresh}: {m.get('iou_at_threshold', 0)*100:.1f}%")

ious = [r.iou for r in results_list]
print(f"\nIoU stats: mean={np.mean(ious):.3f}, std={np.std(ious):.3f}, median={np.median(ious):.3f}")
print(f"Detection rate: {sum(1 for r in results_list if r.found)}/{len(results_list)}")

## Visualization

In [ ]:
import cv2

sorted_results = sorted(results_list, key=lambda r: r.iou)
worst = sorted_results[0]
best = sorted_results[-1]

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

for ax, result, title_prefix in zip(axes, [worst, best], ["Worst", "Best"]):
    img_path = [p for p in all_images if p.name == result.image_name][0]
    img, _ = load_image_and_labels(img_path, LABEL_TEST_DIR)
    vis = img.copy()
    
    if result.gt_bbox:
        x, y, bw, bh = result.gt_bbox
        cv2.rectangle(vis, (x, y), (x+bw, y+bh), (0, 255, 0), 3)
    if result.pred_bbox:
        x, y, bw, bh = result.pred_bbox
        cv2.rectangle(vis, (x, y), (x+bw, y+bh), (0, 0, 255), 2)
    
    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{title_prefix} IoU={result.iou:.3f}")
    ax.axis("off")

plt.suptitle("Green=GT, Red=Pred (YOLO)")
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / "figures" / "yolo_best_worst.png", dpi=150)
plt.show()

## Conclusions

In [ ]:
summary = {
    "model": "YOLOv8n",
    "data": "500 synthetic + 49 real (val)",
    "epochs": 50,
    "train_time_min": round(elapsed/60, 1),
    "iou_mean": round(metrics['iou_mean'], 3),
    "iou_std": round(metrics['iou_std'], 3),
    "precision": round(metrics['precision'], 3),
    "recall": round(metrics['recall'], 3),
    "f1": round(metrics['f1'], 3),
    "detection_rate": round(metrics['detection_rate'], 3),
}

import json
with open(ARTIFACTS_DIR / "metrics" / "yolo_results.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Summary saved to artifacts/metrics/yolo_results.json")
print(json.dumps(summary, indent=2))